# Phase 1.1 Static Map

Load hand-entered static sweep CSVs from `data/raw/`, validate the raw schema, and plot forward/reverse measurements. This notebook is plot-only for Phase 0-1.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
FIG_DIR = ROOT / 'figures'
REQUIRED_COLUMNS = ['pwm_cmd', 'direction', 'vmean_v', 'rpm', 'notes']
VALID_DIRECTIONS = {'fwd', 'rev'}

FIG_DIR.mkdir(exist_ok=True)

In [ ]:
def run_index_from_name(path: Path):
    match = re.search(r'_run(\d+)\.csv$', path.name)
    return int(match.group(1)) if match else None


def load_static_sweep(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if list(df.columns) != REQUIRED_COLUMNS:
        raise ValueError(f'{path.name}: expected columns {REQUIRED_COLUMNS}, got {list(df.columns)}')

    df = df.copy()
    df['source_file'] = path.name
    df['run_index'] = run_index_from_name(path)

    # A copied-but-unfilled template is allowed to exist in data/raw. Skip its
    # incomplete rows instead of treating them as measured data.
    direction_text = df['direction'].astype('string').str.strip().str.lower()
    incomplete = (
        direction_text.isna()
        | (direction_text == '')
        | df['vmean_v'].isna()
        | df['rpm'].isna()
    )
    if incomplete.any():
        print(f'{path.name}: skipped {int(incomplete.sum())} incomplete template rows')

    df = df.loc[~incomplete].copy()
    if df.empty:
        return df

    df['direction'] = df['direction'].astype(str).str.strip().str.lower()
    bad_directions = sorted(set(df['direction']) - VALID_DIRECTIONS)
    if bad_directions:
        raise ValueError(f'{path.name}: invalid direction values {bad_directions}; use fwd or rev')

    for col in ['pwm_cmd', 'vmean_v', 'rpm']:
        df[col] = pd.to_numeric(df[col], errors='raise')

    bad_pwm = df.loc[(df['pwm_cmd'] < 0) | (df['pwm_cmd'] > 255), 'pwm_cmd']
    if not bad_pwm.empty:
        raise ValueError(f'{path.name}: pwm_cmd must stay in 0..255')

    return df


def load_all_static_sweeps(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    paths = sorted(raw_dir.glob('static_sweep_*_run*.csv'))
    if not paths:
        print(f'No static sweep CSVs found in {raw_dir}')
        return pd.DataFrame(columns=REQUIRED_COLUMNS + ['source_file', 'run_index'])

    frames = [load_static_sweep(path) for path in paths]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        print('Static sweep files exist, but no completed rows were found yet.')
        return pd.DataFrame(columns=REQUIRED_COLUMNS + ['source_file', 'run_index'])

    return pd.concat(frames, ignore_index=True)

In [ ]:
df = load_all_static_sweeps()
df.head()

In [ ]:
if df.empty:
    print('No completed static sweep data to plot yet.')
else:
    colors = {'fwd': '#1f77b4', 'rev': '#d62728'}
    labels = {'fwd': 'Forward', 'rev': 'Reverse'}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for direction, group in df.groupby('direction'):
        group = group.sort_values(['run_index', 'pwm_cmd'])
        style = dict(marker='o', linewidth=1.5, markersize=4, color=colors.get(direction))
        axes[0].plot(group['pwm_cmd'], group['rpm'], label=labels.get(direction, direction), **style)
        axes[1].plot(group['pwm_cmd'], group['vmean_v'], label=labels.get(direction, direction), **style)
        axes[2].plot(group['vmean_v'], group['rpm'], label=labels.get(direction, direction), **style)

    axes[0].set_xlabel('PWM command magnitude')
    axes[0].set_ylabel('RPM')
    axes[1].set_xlabel('PWM command magnitude')
    axes[1].set_ylabel('DMM Vmean (V)')
    axes[2].set_xlabel('DMM Vmean (V)')
    axes[2].set_ylabel('RPM')

    for ax in axes:
        ax.grid(True, alpha=0.3)
        ax.legend()

    fig.tight_layout()
    out = FIG_DIR / 'static_map_phase01.png'
    fig.savefig(out, dpi=180)
    print(f'Saved {out}')
    plt.show()